# Day3_08 · 두 수치의 관계 확인

콘크리트 배합자료의 산점도와 상관계수를 연결해 읽습니다.

## 이 노트북에서 확인할 내용

- 산점도에서 두 수치의 관계를 관찰할 수 있습니다.
- Pearson과 Spearman 상관계수를 계산하고 차이를 비교할 수 있습니다.
- 상관관계와 원인 설명을 구분할 수 있습니다.

## 실행 방법

위에서 아래로 코드 셀을 한 개씩 실행하고 바로 다음 결과를 확인합니다. 수정 안내가 있는 곳에서는 표시된 값 하나만 바꾼 뒤 결과를 비교합니다.

### D3B-08-01 · 경로 도구 불러오기

In [ ]:
from pathlib import Path

### D3B-08-02 · 현재 폴더 확인

In [ ]:
current_folder = Path.cwd()

In [ ]:
current_folder

### D3B-08-03 · 데이터 폴더 후보 만들기

In [ ]:
data_candidates = [
    current_folder / 'data',
    current_folder.parent / 'data',
]

In [ ]:
data_candidates

### D3B-08-04 · 데이터 폴더 선택

In [ ]:
DATA_DIR = next(folder for folder in data_candidates if folder.exists())

In [ ]:
DATA_DIR

### D3B-08-05 · pandas 불러오기

In [ ]:
import pandas as pd

### D3B-08-06 · 그림 도구 불러오기

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

### D3B-08-07 · 콘크리트 강도자료 읽기

In [ ]:
concrete = pd.read_csv(DATA_DIR / 'concrete_strength.csv')

In [ ]:
concrete

### D3B-08-08 · 자료 크기 확인

In [ ]:
concrete.shape

### D3B-08-09 · 관계를 볼 열 확인

In [ ]:
concrete[['cement_kg_m3', 'water_kg_m3', 'age_days', 'strength_mpa']].head()

## 두 수치가 함께 움직이는 모습부터 확인하기

점 하나는 콘크리트 배합 한 행입니다. 먼저 그림을 보고 방향과 흩어진 정도를 관찰합니다.

### D3B-08-10 · 시멘트량과 강도 산점도

In [ ]:
sns.scatterplot(data=concrete, x='cement_kg_m3', y='strength_mpa', alpha=0.55)
plt.xlabel('Cement (kg/m³)')
plt.ylabel('Strength (MPa)')
plt.title('Cement and concrete strength')
plt.show()

### D3B-08-11 · 시멘트량과 강도의 Pearson 값 계산

In [ ]:
cement_pearson = concrete['cement_kg_m3'].corr(concrete['strength_mpa'], method='pearson')

In [ ]:
cement_pearson

### D3B-08-12 · 시멘트량과 강도의 Spearman 값 계산

In [ ]:
cement_spearman = concrete['cement_kg_m3'].corr(concrete['strength_mpa'], method='spearman')

In [ ]:
cement_spearman

### D3B-08-13 · 두 결과를 한 표로 비교

In [ ]:
cement_correlation = pd.Series({'Pearson': cement_pearson, 'Spearman': cement_spearman}, name='correlation')

In [ ]:
cement_correlation.round(3)

Pearson 값은 두 수치의 직선적인 관계를, Spearman 값은 값의 순서가 함께 오르내리는 관계를 요약합니다. 두 값 모두 -1에서 1 사이이며, 0에 가깝다고 해서 모든 형태의 관계가 없다는 뜻은 아닙니다.

### D3B-08-14 · 재령과 강도 산점도

In [ ]:
sns.scatterplot(data=concrete, x='age_days', y='strength_mpa', alpha=0.55)
plt.xlabel('Age (days)')
plt.ylabel('Strength (MPa)')
plt.title('Age and concrete strength')
plt.show()

### D3B-08-15 · 재령과 강도의 두 상관계수 계산

In [ ]:
age_correlation = pd.Series({
    'Pearson': concrete['age_days'].corr(concrete['strength_mpa'], method='pearson'),
    'Spearman': concrete['age_days'].corr(concrete['strength_mpa'], method='spearman'),
})

In [ ]:
age_correlation.round(3)

### D3B-08-16 · 물 사용량과 강도 산점도

In [ ]:
sns.scatterplot(data=concrete, x='water_kg_m3', y='strength_mpa', alpha=0.55)
plt.xlabel('Water (kg/m³)')
plt.ylabel('Strength (MPa)')
plt.title('Water and concrete strength')
plt.show()

### D3B-08-17 · 수치형 열의 상관표 만들기

In [ ]:
correlation_table = concrete.corr(numeric_only=True)

In [ ]:
correlation_table.round(2)

### D3B-08-18 · 상관표를 색으로 보기

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(correlation_table, cmap='vlag', center=0, annot=True, fmt='.2f')
plt.title('Correlation table of concrete variables')
plt.show()

### D3B-08-19 · 강도와의 상관계수만 정렬

In [ ]:
strength_correlation = correlation_table['strength_mpa'].drop('strength_mpa').sort_values()

In [ ]:
strength_correlation.round(3)

상관계수의 부호는 함께 움직이는 방향을 나타냅니다. 상관계수는 다른 배합조건을 동시에 통제하지 않으며, 원인과 결과를 증명하지 않습니다.

### D3B-08-20 · 확인할 입력 열 선택

In [ ]:
# 아래 열 이름을 다른 수치형 열로 바꾸어 보세요.
x_column = 'slag_kg_m3'

In [ ]:
x_column

### D3B-08-21 · 선택한 열과 강도 산점도

In [ ]:
sns.scatterplot(data=concrete, x=x_column, y='strength_mpa', alpha=0.55)
plt.title(f'{x_column} and concrete strength')
plt.show()

### D3B-08-22 · 선택한 열의 두 상관계수 계산

In [ ]:
selected_correlation = pd.Series({
    'Pearson': concrete[x_column].corr(concrete['strength_mpa'], method='pearson'),
    'Spearman': concrete[x_column].corr(concrete['strength_mpa'], method='spearman'),
})

In [ ]:
selected_correlation.round(3)

### D3B-08-23 · 해석 문장 틀 만들기

In [ ]:
correlation_note = pd.Series({
    '그림에서 본 방향': '증가 / 감소 / 뚜렷하지 않음 중 선택',
    'Pearson': round(selected_correlation['Pearson'], 3),
    'Spearman': round(selected_correlation['Spearman'], 3),
    '말할 수 없는 것': '이 변수의 변화가 강도 변화의 원인이라는 결론',
})

In [ ]:
correlation_note

### D3B-08-CODEX · Codex에 확장 분석 요청하기

In [ ]:
codex_request = '''이 Notebook의 기존 셀은 수정하지 마세요. concrete DataFrame과 x_column을 사용하세요. 선택한 열과 strength_mpa의 관계를 보여 주는 그래프 한 개를 새 코드 셀로 제안해 주세요. 그래프 아래에 Pearson과 Spearman 값이 다른 이유를 데이터 모양과 연결해 두 문장으로 설명해 주세요. 인과관계로 표현하지 마세요.'''.strip()

In [ ]:
codex_request

### D3B-08-CHECK · 제안받은 코드를 실행하기 전에 확인하기

In [ ]:
codex_checklist = pd.Series(['기존 concrete를 덮어쓰지 않는가?', '두 축과 단위가 보이는가?', '상관관계를 원인으로 단정하지 않는가?'], name='확인할 내용')

In [ ]:
codex_checklist